# Análisis del catálogo CISA KEV

Proveedores más afectados, uso en ransomware y tendencia mensual.

In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt

URL = 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
RUTA = Path('../data/kev.json')

if not RUTA.exists():
    RUTA.parent.mkdir(exist_ok=True)
    RUTA.write_bytes(requests.get(URL, timeout=60).content)

kev = json.loads(RUTA.read_text(encoding='utf-8'))
df = pd.DataFrame(kev['vulnerabilities'])
df['dateAdded'] = pd.to_datetime(df['dateAdded'])
print(kev['catalogVersion'], len(df), 'vulnerabilidades')
df.head()

## Proveedores más afectados

In [ ]:
top = df['vendorProject'].value_counts().head(10)
top.sort_values().plot.barh(title='Top 10 proveedores en KEV')
plt.tight_layout(); plt.show()

## Uso conocido en ransomware

In [ ]:
pct = (df['knownRansomwareCampaignUse'] == 'Known').mean() * 100
print(f'{pct:.1f} % de las vulnerabilidades KEV se usan en campañas de ransomware')

## Tendencia mensual de incorporaciones

In [ ]:
mensual = df.set_index('dateAdded').resample('MS').size()
mensual.plot(title='Vulnerabilidades agregadas a KEV por mes')
plt.tight_layout(); plt.show()